In [3]:
# Task 1: Build a Robust NLP Preprocessing Engine (Advanced)

import re
import string
import nltk
from collections import Counter

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [4]:
# TASK 1: Conceptual Understanding (Written Answers)

print("=" * 70)
print("TASK 1: CONCEPTUAL UNDERSTANDING")
print("=" * 70)

print("""
Q1: What is the difference between "Love" and "love" in NLP?
─────────────────────────────────────────────────────────────
Answer:
In NLP, "Love" and "love" are treated as two DIFFERENT tokens by default
because most programming languages and NLP tools are case-sensitive.

- "Love" (uppercase L) and "love" (lowercase l) have different ASCII values.
- In a bag-of-words model or TF-IDF, they would be counted as separate words.
- This inflates the vocabulary size unnecessarily.
- For example, "Love", "LOVE", "love" would create 3 separate entries
  in a word frequency dictionary, even though they carry the same meaning.
- Solution: We convert all text to lowercase during preprocessing to
  ensure consistency and reduce vocabulary size.
- Exception: In Named Entity Recognition (NER), case matters because
  "Apple" (company) vs "apple" (fruit) carry different meanings.

────────────────────────────────────────────────────────────────────────

Q2: What happens if stopwords are not removed?
──────────────────────────────────────────────
Answer:
Stopwords are common words like "the", "is", "at", "and", "a", "in", etc.
If stopwords are NOT removed:

1. Feature Space Inflation: The feature vectors become very large with
   irrelevant dimensions.
2. Noise Domination: High-frequency stopwords dominate the analysis,
   masking actually important words.
3. Poor Model Performance: ML models may learn patterns from these
   common words rather than meaningful content words.
4. Increased Computation: More tokens mean more processing time and
   memory usage.
5. Misleading TF-IDF Scores: Stopwords will have high term frequency
   but low discriminative power, skewing results.

Example: In the sentence "This is the best movie", the words "this",
"is", "the" add no value for sentiment analysis. The word "best" is
what carries the sentiment.

────────────────────────────────────────────────────────────────────────

Q3: Provide two real-world scenarios where removing stopwords can be harmful.
─────────────────────────────────────────────────────────────────────────────
Answer:

Scenario 1: Sentiment Analysis with Negation
- Sentence: "I am NOT happy with this service"
- If we remove stopwords (including "not"), the sentence becomes:
  "happy service" — which completely REVERSES the sentiment!
- The word "not" is a stopword but carries critical meaning for
  sentiment analysis. Removing it changes negative to positive.

Scenario 2: Question Answering / Search Engines
- Query: "To be or not to be"
- If stopwords are removed: "" (empty string — everything is a stopword!)
- The entire meaning of Shakespeare's famous quote is lost.
- Another example: "What is the capital of France?"
  Removing stopwords → "capital France" — loses the question structure.
- Search engines need to understand "how to" vs "what is" vs "where is"
  to return relevant results.

────────────────────────────────────────────────────────────────────────

Q4: Explain the difference between stemming and lemmatization.
──────────────────────────────────────────────────────────────
Answer:

STEMMING:
- Stemming is a rule-based process that chops off word suffixes to
  get the root form (called a "stem").
- It uses heuristic rules (like removing "ing", "ed", "ly", "s").
- It is FAST but can produce non-real words.
- Example using Porter Stemmer:
    "running"  → "run"
    "better"   → "better" (cannot handle irregular words)
    "studies"  → "studi" (NOT a real word!)
    "fairly"   → "fairli" (NOT a real word!)

LEMMATIZATION:
- Lemmatization uses vocabulary and morphological analysis to return
  the base dictionary form of a word (called a "lemma").
- It considers the Part of Speech (POS) of the word.
- It is SLOWER but always produces valid, real words.
- Example using WordNet Lemmatizer:
    "running"  → "run"
    "better"   → "good" (handles irregular words!)
    "studies"  → "study" (valid word!)
    "fairly"   → "fairly"

Key Differences:
┌──────────────┬───────────────────┬────────────────────┐
│   Aspect     │    Stemming       │   Lemmatization    │
├──────────────┼───────────────────┼────────────────────┤
│ Approach     │ Rule-based        │ Dictionary-based   │
│ Speed        │ Faster            │ Slower             │
│ Output       │ May not be real   │ Always real word   │
│ Accuracy     │ Less accurate     │ More accurate      │
│ POS needed?  │ No                │ Yes (optional)     │
│ Example      │ "caring"→"car"    │ "caring"→"care"    │
└──────────────┴───────────────────┴────────────────────┘
""")

TASK 1: CONCEPTUAL UNDERSTANDING

Q1: What is the difference between "Love" and "love" in NLP?
─────────────────────────────────────────────────────────────
Answer:
In NLP, "Love" and "love" are treated as two DIFFERENT tokens by default
because most programming languages and NLP tools are case-sensitive.

- "Love" (uppercase L) and "love" (lowercase l) have different ASCII values.
- In a bag-of-words model or TF-IDF, they would be counted as separate words.
- This inflates the vocabulary size unnecessarily.
- For example, "Love", "LOVE", "love" would create 3 separate entries
  in a word frequency dictionary, even though they carry the same meaning.
- Solution: We convert all text to lowercase during preprocessing to
  ensure consistency and reduce vocabulary size.
- Exception: In Named Entity Recognition (NER), case matters because
  "Apple" (company) vs "apple" (fruit) carry different meanings.

────────────────────────────────────────────────────────────────────────

Q2: What happe

In [6]:
# TASK 2: Build Advanced Preprocessing Function

print("=" * 70)
print("TASK 2: BUILD ADVANCED PREPROCESSING FUNCTION")
print("=" * 70)

def preprocess_text(text):
    """
    Advanced NLP preprocessing function that transforms raw, noisy text
    into clean, structured tokens.
    
    Parameters:
        text (str): Raw input text
    
    Returns:
        dict: Dictionary containing 'original', 'cleaned_tokens', 'cleaned_sentence'
    """
    
    # ---- Error Handling ----
    # Handle None input
    if text is None:
        return {
            'original': None,
            'cleaned_tokens': [],
            'cleaned_sentence': ''
        }
    
    # Handle non-string input
    if not isinstance(text, str):
        return {
            'original': str(text),
            'cleaned_tokens': [],
            'cleaned_sentence': ''
        }
    
    # Handle empty string
    if text.strip() == '':
        return {
            'original': text,
            'cleaned_tokens': [],
            'cleaned_sentence': ''
        }
    
    # Store original text
    original = text
    
    # ---- Step 1: Remove URLs ----
    # Matches http, https, www patterns
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    
    # ---- Step 2: Remove email-like patterns ----
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    
    # ---- Step 3: Remove emojis ----
    # Remove emoji characters using Unicode ranges
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002702-\U000027B0"  # dingbats
        u"\U000024C2-\U0001F251"  # enclosed characters
        u"\U0001f926-\U0001f937"  # additional emoticons
        u"\U00010000-\U0010ffff"  # supplementary characters
        "]+", 
        flags=re.UNICODE
    )
    text = emoji_pattern.sub('', text)
    
    # ---- Step 4: Convert to lowercase ----
    text = text.lower()
    
    # ---- Step 5: Handle repeated characters ----
    # Reduce 3 or more consecutive same characters to 2, then to 1
    # Example: "soooo" → "so", "goooood" → "god" → we handle smartly
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # ---- Step 6: Remove special characters and punctuation ----
    # Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # ---- Step 7: Remove numbers (already handled by step 6) ----
    # Numbers are already removed since we only keep alphabetic chars
    
    # ---- Step 8: Remove extra spaces ----
    text = re.sub(r'\s+', ' ', text).strip()
    
    # ---- Step 9: Tokenize ----
    tokens = text.split()
    
    # ---- Step 10: Remove very short tokens (length <= 2) ----
    # Exception: keep meaningful short words like "no", "not"
    meaningful_short_words = {'no', 'not', 'nor', 'do', 'am', 'is', 'be', 
                               'he', 'me', 'we', 'us', 'so', 'if', 'or',
                               'an', 'as', 'at', 'by', 'up', 'go', 'my',
                               'on', 'to', 'in', 'it', 'of'}
    
    tokens = [token for token in tokens 
              if len(token) > 2 or token in meaningful_short_words]
    
    # ---- Step 11: Build cleaned sentence ----
    cleaned_sentence = ' '.join(tokens)
    
    return {
        'original': original,
        'cleaned_tokens': tokens,
        'cleaned_sentence': cleaned_sentence
    }

# Test the function with a simple example
test_result = preprocess_text("Hello World!!! This is a TEST 123")
print(f"\nFunction created successfully!")
print(f"Test Input:  'Hello World!!! This is a TEST 123'")
print(f"Test Output: {test_result}")

TASK 2: BUILD ADVANCED PREPROCESSING FUNCTION

Function created successfully!
Test Input:  'Hello World!!! This is a TEST 123'
Test Output: {'original': 'Hello World!!! This is a TEST 123', 'cleaned_tokens': ['hello', 'world', 'this', 'is', 'test'], 'cleaned_sentence': 'hello world this is test'}


In [7]:
# TASK 3: Stress Testing with 10 Diverse Sentences

print("=" * 70)
print("TASK 3: STRESS TESTING")
print("=" * 70)

# Define 10 diverse test sentences
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# Process each sentence and store results
all_results = []

for i, sentence in enumerate(test_sentences, 1):
    result = preprocess_text(sentence)
    all_results.append(result)
    
    print(f"\n{'─' * 60}")
    print(f"Test Case {i}:")
    print(f"{'─' * 60}")
    print(f"  Original Text   : {result['original']}")
    print(f"  Cleaned Tokens  : {result['cleaned_tokens']}")
    print(f"  Cleaned Sentence: {result['cleaned_sentence']}")

print(f"\n{'═' * 60}")
print(f"All {len(test_sentences)} test cases processed successfully!")
print(f"{'═' * 60}")

TASK 3: STRESS TESTING

────────────────────────────────────────────────────────────
Test Case 1:
────────────────────────────────────────────────────────────
  Original Text   : Get 100% FREE access now!!!
  Cleaned Tokens  : ['get', 'free', 'access', 'now']
  Cleaned Sentence: get free access now

────────────────────────────────────────────────────────────
Test Case 2:
────────────────────────────────────────────────────────────
  Original Text   : I absolutely looooved this product 😍😍
  Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
  Cleaned Sentence: absolutely loved this product

────────────────────────────────────────────────────────────
Test Case 3:
────────────────────────────────────────────────────────────
  Original Text   : Worst service ever... 0/10
  Cleaned Tokens  : ['worst', 'service', 'ever']
  Cleaned Sentence: worst service ever

────────────────────────────────────────────────────────────
Test Case 4:
───────────────────────────────────────────────

In [9]:
# TASK 4: Token Analytics

print("=" * 70)
print("TASK 4: TOKEN ANALYTICS")
print("=" * 70)

# Compute analytics for each sentence
analytics_data = []

print(f"\n{'─' * 80}")
print(f"{'Sentence':<6} {'Total Tokens':<15} {'Unique Tokens':<16} {'Avg Token Len':<15}")
print(f"{'─' * 80}")

for i, result in enumerate(all_results, 1):
    tokens = result['cleaned_tokens']
    
    # Total number of tokens
    total_tokens = len(tokens)
    
    # Number of unique tokens
    unique_tokens = len(set(tokens))
    
    # Average token length
    if total_tokens > 0:
        avg_length = round(sum(len(t) for t in tokens) / total_tokens, 2)
    else:
        avg_length = 0
    
    analytics = {
        'sentence_num': i,
        'original': result['original'],
        'total_tokens': total_tokens,
        'unique_tokens': unique_tokens,
        'avg_token_length': avg_length,
        'cleaned_tokens': tokens
    }
    analytics_data.append(analytics)
    
    print(f"  {i:<6} {total_tokens:<15} {unique_tokens:<16} {avg_length:<15}")

print(f"{'─' * 80}")

# ---- Analysis Questions ----
print(f"\n{'=' * 70}")
print("ANALYSIS QUESTIONS")
print(f"{'=' * 70}")

# Q1: Which sentence had the most noise?
# Noise = difference between original word count and cleaned token count
print("\nQ1: Which sentence had the most noise?")
print("─" * 50)

noise_analysis = []
for i, result in enumerate(all_results, 1):
    original_length = len(result['original'])
    cleaned_length = len(result['cleaned_sentence'])
    original_word_count = len(result['original'].split())
    cleaned_word_count = len(result['cleaned_tokens'])
    
    # Calculate noise as characters removed + tokens removed
    char_noise = original_length - cleaned_length
    token_noise = original_word_count - cleaned_word_count
    noise_ratio = round((1 - cleaned_length / max(original_length, 1)) * 100, 2)
    
    noise_analysis.append({
        'sentence_num': i,
        'original': result['original'],
        'char_noise': char_noise,
        'token_noise': token_noise,
        'noise_ratio': noise_ratio
    })
    
    print(f"  Sentence {i}: Noise Ratio = {noise_ratio}% "
          f"(Chars removed: {char_noise}, Tokens removed: {token_noise})")

# Find the noisiest sentence
noisiest = max(noise_analysis, key=lambda x: x['noise_ratio'])
print(f"\n  → ANSWER: Sentence {noisiest['sentence_num']} had the MOST noise "
      f"({noisiest['noise_ratio']}% noise)")
print(f"    Original: \"{noisiest['original']}\"")

# Q2: Which sentence retained the most meaningful tokens?
print(f"\nQ2: Which sentence retained the most meaningful tokens after cleaning?")
print("─" * 50)

for data in analytics_data:
    original_words = len(all_results[data['sentence_num']-1]['original'].split())
    retention = round((data['total_tokens'] / max(original_words, 1)) * 100, 2)
    print(f"  Sentence {data['sentence_num']}: Retained {data['total_tokens']}/{original_words} "
          f"tokens ({retention}%)")

most_retained = max(analytics_data, key=lambda x: x['total_tokens'])
print(f"\n  → ANSWER: Sentence {most_retained['sentence_num']} retained the MOST "
      f"meaningful tokens ({most_retained['total_tokens']} tokens)")
print(f"    Original: \"{all_results[most_retained['sentence_num']-1]['original']}\"")

TASK 4: TOKEN ANALYTICS

────────────────────────────────────────────────────────────────────────────────
Sentence Total Tokens    Unique Tokens    Avg Token Len  
────────────────────────────────────────────────────────────────────────────────
  1      4               4                4.0            
  2      4               4                6.5            
  3      3               3                5.33           
  4      3               3                2.67           
  5      5               5                3.8            
  6      2               2                4.0            
  7      4               4                2.75           
  8      2               2                2.5            
  9      4               4                4.5            
  10     5               5                3.6            
────────────────────────────────────────────────────────────────────────────────

ANALYSIS QUESTIONS

Q1: Which sentence had the most noise?
──────────────────────────────────

In [13]:
# TASK 5: Frequency Analysis

print("=" * 70)
print("TASK 5: FREQUENCY ANALYSIS")
print("=" * 70)

# Combine all tokens from all processed sentences
all_tokens = []
for result in all_results:
    all_tokens.extend(result['cleaned_tokens'])

print(f"\nTotal tokens across all sentences: {len(all_tokens)}")
print(f"Total unique tokens: {len(set(all_tokens))}")

# Use Counter for frequency analysis
token_counter = Counter(all_tokens)

# Top 10 most frequent words
print(f"\n{'─' * 50}")
print("TOP 10 MOST FREQUENT WORDS")
print(f"{'─' * 50}")
print(f"{'Rank':<6} {'Word':<20} {'Frequency':<10}")
print(f"{'─' * 50}")

top_10 = token_counter.most_common(10)
for rank, (word, freq) in enumerate(top_10, 1):
    bar = '█' * freq
    print(f"  {rank:<6} {word:<20} {freq:<10} {bar}")

# Top 5 least frequent words
print(f"\n{'─' * 50}")
print("TOP 5 LEAST FREQUENT WORDS")
print(f"{'─' * 50}")
print(f"{'Rank':<6} {'Word':<20} {'Frequency':<10}")
print(f"{'─' * 50}")

least_5 = token_counter.most_common()[:-6:-1]  # Get last 5 items (least frequent)
for rank, (word, freq) in enumerate(least_5, 1):
    bar = '░' * freq
    print(f"  {rank:<6} {word:<20} {freq:<10} {bar}")

# Full frequency distribution
print(f"\n{'─' * 50}")
print("COMPLETE FREQUENCY DISTRIBUTION")
print(f"{'─' * 50}")
for word, freq in sorted(token_counter.items(), key=lambda x: -x[1]):
    print(f"  {word:<20} : {freq}")

TASK 5: FREQUENCY ANALYSIS

Total tokens across all sentences: 36
Total unique tokens: 30

──────────────────────────────────────────────────
TOP 10 MOST FREQUENT WORDS
──────────────────────────────────────────────────
Rank   Word                 Frequency 
──────────────────────────────────────────────────
  1      this                 4          ████
  2      now                  3          ███
  3      is                   2          ██
  4      get                  1          █
  5      free                 1          █
  6      access               1          █
  7      absolutely           1          █
  8      loved                1          █
  9      product              1          █
  10     worst                1          █

──────────────────────────────────────────────────
TOP 5 LEAST FREQUENT WORDS
──────────────────────────────────────────────────
Rank   Word                 Frequency 
──────────────────────────────────────────────────
  1      with                 1   

In [17]:
# TASK 6: Build Full Pipeline

print("=" * 70)
print("TASK 6: FULL PIPELINE")
print("=" * 70)

def full_pipeline(text_list):
    """
    Complete NLP preprocessing pipeline that processes a list of texts
    and returns cleaned tokens and sentences.
    
    Parameters:
        text_list (list): List of raw text strings
    
    Returns:
        dict: Dictionary with 'tokens' (list of token lists) and 
              'clean_sentences' (list of cleaned sentences)
    """
    
    # Validate input
    if not isinstance(text_list, list):
        print("Error: Input must be a list of strings.")
        return {"tokens": [], "clean_sentences": []}
    
    if len(text_list) == 0:
        print("Warning: Empty list provided.")
        return {"tokens": [], "clean_sentences": []}
    
    all_tokens = []
    all_clean_sentences = []
    
    for text in text_list:
        result = preprocess_text(text)
        all_tokens.append(result['cleaned_tokens'])
        all_clean_sentences.append(result['cleaned_sentence'])
    
    return {
        "tokens": all_tokens,
        "clean_sentences": all_clean_sentences
    }

# Test the full pipeline with the sample inputs
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# Run the pipeline
pipeline_output = full_pipeline(sample_inputs)

# Display results
print(f"\n{'─' * 60}")
print("PIPELINE OUTPUT")
print(f"{'─' * 60}")

print("\n--- Tokens ---")
for i, tokens in enumerate(pipeline_output['tokens'], 1):
    print(f"  Sentence {i}: {tokens}")

print("\n--- Clean Sentences ---")
for i, sentence in enumerate(pipeline_output['clean_sentences'], 1):
    print(f"  Sentence {i}: \"{sentence}\"")

# Display in the exact required format
print(f"\n{'─' * 60}")
print("OUTPUT IN REQUIRED FORMAT:")
print(f"{'─' * 60}")
print("{")
print(f'    "tokens": {pipeline_output["tokens"]},')
print(f'    "clean_sentences": {pipeline_output["clean_sentences"]}')
print("}")

TASK 6: FULL PIPELINE

────────────────────────────────────────────────────────────
PIPELINE OUTPUT
────────────────────────────────────────────────────────────

--- Tokens ---
  Sentence 1: ['get', 'free', 'access', 'now']
  Sentence 2: ['absolutely', 'loved', 'this', 'product']
  Sentence 3: ['worst', 'service', 'ever']
  Sentence 4: ['call', 'me', 'at']
  Sentence 5: ['this', 'is', 'the', 'best', 'course']
  Sentence 6: ['visit', 'now']
  Sentence 7: ['no', 'this', 'is', 'bad']
  Sentence 8: ['got', 'it']
  Sentence 9: ['win', 'now', 'limited', 'offer']
  Sentence 10: ['am', 'not', 'happy', 'with', 'this']

--- Clean Sentences ---
  Sentence 1: "get free access now"
  Sentence 2: "absolutely loved this product"
  Sentence 3: "worst service ever"
  Sentence 4: "call me at"
  Sentence 5: "this is the best course"
  Sentence 6: "visit now"
  Sentence 7: "no this is bad"
  Sentence 8: "got it"
  Sentence 9: "win now limited offer"
  Sentence 10: "am not happy with this"

───────────────

In [18]:
# TASK 7: Error Handling

print("=" * 70)
print("TASK 7: ERROR HANDLING")
print("=" * 70)

# Define edge case test inputs
error_test_cases = [
    {
        'name': 'Empty String',
        'input': ''
    },
    {
        'name': 'Only Emojis',
        'input': '😍😍😍🎉🎊💯🔥'
    },
    {
        'name': 'Only Numbers',
        'input': '123 456 789 000'
    },
    {
        'name': 'None Input',
        'input': None
    },
    {
        'name': 'Only Spaces',
        'input': '          '
    },
    {
        'name': 'Only Special Characters',
        'input': '!@#$%^&*()_+{}|:<>?'
    },
    {
        'name': 'Only URLs',
        'input': 'https://google.com https://openai.com'
    },
    {
        'name': 'Mixed Noise (Numbers + Emojis + Special Chars)',
        'input': '123!!! 😍😍 $$$$ ###'
    },
    {
        'name': 'Single Character',
        'input': 'a'
    },
    {
        'name': 'Very Long Repeated Characters',
        'input': 'aaaaaaaaaaaaaaaaaaaabbbbbbbbbbbbccccccccc'
    }
]

# Test each edge case
for i, test_case in enumerate(error_test_cases, 1):
    print(f"\n{'─' * 60}")
    print(f"Edge Case {i}: {test_case['name']}")
    print(f"{'─' * 60}")
    
    try:
        result = preprocess_text(test_case['input'])
        print(f"  Input          : {repr(test_case['input'])}")
        print(f"  Cleaned Tokens : {result['cleaned_tokens']}")
        print(f"  Cleaned Text   : \"{result['cleaned_sentence']}\"")
        print(f"  Status         : ✅ Handled Successfully")
    except Exception as e:
        print(f"  Input          : {repr(test_case['input'])}")
        print(f"  Error          : {str(e)}")
        print(f"  Status         : ❌ Error Occurred")

# Test full pipeline with edge cases
print(f"\n{'═' * 60}")
print("Testing Full Pipeline with Edge Cases")
print(f"{'═' * 60}")

edge_case_list = ['', '😍😍😍', '123 456', None, '   ']

# Filter None values for pipeline (since pipeline expects strings)
safe_list = [text if text is not None else '' for text in edge_case_list]

try:
    edge_pipeline_result = full_pipeline(safe_list)
    print(f"\n  Pipeline handled all edge cases successfully! ✅")
    print(f"  Tokens: {edge_pipeline_result['tokens']}")
    print(f"  Clean Sentences: {edge_pipeline_result['clean_sentences']}")
except Exception as e:
    print(f"  Pipeline Error: {str(e)} ❌")

print(f"\n{'═' * 60}")
print("ALL TASKS COMPLETED SUCCESSFULLY! ✅")
print(f"{'═' * 60}")

TASK 7: ERROR HANDLING

────────────────────────────────────────────────────────────
Edge Case 1: Empty String
────────────────────────────────────────────────────────────
  Input          : ''
  Cleaned Tokens : []
  Cleaned Text   : ""
  Status         : ✅ Handled Successfully

────────────────────────────────────────────────────────────
Edge Case 2: Only Emojis
────────────────────────────────────────────────────────────
  Input          : '😍😍😍🎉🎊💯🔥'
  Cleaned Tokens : []
  Cleaned Text   : ""
  Status         : ✅ Handled Successfully

────────────────────────────────────────────────────────────
Edge Case 3: Only Numbers
────────────────────────────────────────────────────────────
  Input          : '123 456 789 000'
  Cleaned Tokens : []
  Cleaned Text   : ""
  Status         : ✅ Handled Successfully

────────────────────────────────────────────────────────────
Edge Case 4: None Input
────────────────────────────────────────────────────────────
  Input          : None
  Cleaned Toke